In [64]:
import numpy as np
import pandas as pd
from pylab import plt, mpl
from sklearn.metrics import accuracy_score
import os
import papermill
import talib as ta
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential

frequency = "1d"
window_pred = 7

# Cargar los datos para esta frecuencia
BTCUSDT = pd.read_csv('BTCUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
'''BTCUSDT4h = pd.read_csv('BTCUSDT_4h_01-01-2016_01-01-2025.csv', index_col='timestamp')'''
'''BTCUSDTh = pd.read_csv('BTCUSDT_1h_01-01-2016_01-01-2025.csv', index_col='timestamp')'''
AVAXUSDT = pd.read_csv('AVAXUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
ETHUSDT = pd.read_csv('ETHUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
data = pd.DataFrame() 

data['BTCUSDT'] = BTCUSDT[['close']]
'''data['BTCUSDT4h'] = BTCUSDT4h[['close']]'''
data['AVAXUSDT'] = AVAXUSDT[['close']]
data['ETHUSDT'] = ETHUSDT[['close']]
data.dropna(inplace=True)
data.head()

,BTCUSDT,AVAXUSDT,ETHUSDT
timestamp,,,
2020-09-22,10529.61,5.3193,344.21
2020-09-23,10241.46,3.5350,320.72
2020-09-24,10736.32,4.6411,348.97
2020-09-25,10686.67,4.7134,351.92
2020-09-26,10728.60,4.5200,353.92


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [65]:
def save_results(model, ric, acc, sample, frequency=frequency):
    # Verificar si el archivo ya existe
    file_name = f'accuracy_results_{frequency}_charac.csv'

    # Si el archivo existe, leer los datos previos, si no, crear un nuevo DataFrame vacío
    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])

    # Agregar la nueva fila con los resultados
    new_row = pd.DataFrame([[model, ric, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)

    # Guardar los resultados acumulados
    df_results.to_csv(file_name, index=False) 

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [66]:
def add_lags(data, ric, lags, window_pred, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df[ric].pct_change(window) #momentum de la ventana pct_change(12)
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df.dropna(inplace=True)
    df = df.iloc[:-window_pred]
    df['d'] = np.where(df[ric].shift(-window_pred) > df[ric], 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán
    features = [ric, 'r', 'd', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

lags = 5

dfs = {}
for ric in data:
    df, cols = add_lags(data, ric, lags, window_pred)
    dfs[ric] = df.dropna(), cols

In [67]:
dfs[ric][0]

,ETHUSDT,r,sma,min,max,mom,vol,rsi,atr,d,...,rsi_lag_1,rsi_lag_2,rsi_lag_3,rsi_lag_4,rsi_lag_5,atr_lag_1,atr_lag_2,atr_lag_3,atr_lag_4,atr_lag_5
timestamp,,,,,,,,,,,,,,,,,,,,,
2020-10-27,403.45,0.027465,372.600667,340.75,413.98,0.128563,0.024947,60.295407,8.020146,0,...,58.405901,61.880238,63.601973,63.123426,64.439753,7.919806,7.732903,7.783003,7.946900,8.053000
2020-10-28,388.23,-0.038455,373.743000,340.75,413.98,0.096819,0.026018,56.592096,8.260141,1,...,60.295407,58.405901,61.880238,63.601973,63.123426,8.020146,7.919806,7.732903,7.783003,7.946900
2020-10-29,387.13,-0.002837,374.654333,340.75,413.98,0.075989,0.025917,56.333410,8.021470,1,...,56.592096,60.295407,58.405901,61.880238,63.601973,8.260141,8.020146,7.919806,7.732903,7.783003
2020-10-30,382.49,-0.012058,375.409667,340.75,413.98,0.062974,0.026050,55.231732,7.908754,1,...,56.333410,56.592096,60.295407,58.405901,61.880238,8.021470,8.260141,8.020146,7.919806,7.732903
2020-10-31,386.46,0.010326,376.530000,340.75,413.98,0.095253,0.025764,55.993463,7.777463,1,...,55.231732,56.333410,56.592096,60.295407,58.405901,7.908754,8.021470,8.260141,8.020146,7.919806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-21,3338.92,-0.039144,3682.060333,3324.73,4004.15,-0.005033,0.037114,49.401226,94.659519,0,...,51.834147,50.865433,54.923680,60.881956,63.203130,93.327433,94.642172,90.671557,84.618508,84.321560
2024-12-22,3281.83,-0.017246,3680.528667,3281.83,4004.15,-0.013808,0.037217,48.394767,93.407201,0,...,49.401226,51.834147,50.865433,54.923680,60.881956,94.659519,93.327433,94.642172,90.671557,84.618508
2024-12-23,3422.53,0.041979,3681.482667,3281.83,4004.15,0.008433,0.037851,50.942875,94.983628,0,...,48.394767,49.401226,51.834147,50.865433,54.923680,93.407201,94.659519,93.327433,94.642172,90.671557


In [68]:
'''from sklearn.neural_network import MLPClassifier

for ric in data:
    model = MLPClassifier(hidden_layer_sizes=[512],
                        random_state=100,
                        max_iter=1000,
                        early_stopping=True,
                        validation_fraction=0.15,
                        shuffle=False)
    df, cols = dfs[ric]
    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()
    model.fit(df[cols], df['d'])
    pred = model.predict(df[cols])
    acc = accuracy_score(df['d'], pred)
    print(f'IN-SAMPLE | {ric:7s} | acc={acc:.4f}')
    save_results('MLPClassifier', ric, acc, "IN-SAMPLE")
'''


'from sklearn.neural_network import MLPClassifier\n\nfor ric in data:\n    model = MLPClassifier(hidden_layer_sizes=[512],\n                        random_state=100,\n                        max_iter=1000,\n                        early_stopping=True,\n                        validation_fraction=0.15,\n                        shuffle=False)\n    df, cols = dfs[ric]\n    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()\n    model.fit(df[cols], df[\'d\'])\n    pred = model.predict(df[cols])\n    acc = accuracy_score(df[\'d\'], pred)\n    print(f\'IN-SAMPLE | {ric:7s} | acc={acc:.4f}\')\n    save_results(\'MLPClassifier\', ric, acc, "IN-SAMPLE")\n'

In [69]:
'''import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential

np.random.seed(100)
tf.random.set_seed(100)

def create_model(problem='regression'):
    model = Sequential()
    model.add(Dense(512, input_dim=len(cols),
                    activation='relu'))
    if problem == 'regression':
        model.add(Dense(1, activation='linear'))
        model.compile(loss='mse', optimizer='adam')
    else:
        model.add(Dense(1, activation='sigmoid'))
        model.compile(loss='binary_crossentropy', optimizer='adam')
    return model

for ric in data:
    model = create_model('classification')
    df, cols = dfs[ric]
    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()
    model.fit(df[cols], df['d'], epochs=50, verbose=False)
    pred = np.where(model.predict(df[cols]) > 0.5, 1, 0)
    acc = accuracy_score(df['d'], pred)
    print(f'IN-SAMPLE | {ric:7s} | acc={acc:.4f}')
    save_results('classification', ric, acc, "IN-SAMPLE")'''

'import tensorflow as tf\nfrom keras.layers import Dense\nfrom keras.models import Sequential\n\nnp.random.seed(100)\ntf.random.set_seed(100)\n\ndef create_model(problem=\'regression\'):\n    model = Sequential()\n    model.add(Dense(512, input_dim=len(cols),\n                    activation=\'relu\'))\n    if problem == \'regression\':\n        model.add(Dense(1, activation=\'linear\'))\n        model.compile(loss=\'mse\', optimizer=\'adam\')\n    else:\n        model.add(Dense(1, activation=\'sigmoid\'))\n        model.compile(loss=\'binary_crossentropy\', optimizer=\'adam\')\n    return model\n\nfor ric in data:\n    model = create_model(\'classification\')\n    df, cols = dfs[ric]\n    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()\n    model.fit(df[cols], df[\'d\'], epochs=50, verbose=False)\n    pred = np.where(model.predict(df[cols]) > 0.5, 1, 0)\n    acc = accuracy_score(df[\'d\'], pred)\n    print(f\'IN-SAMPLE | {ric:7s} | acc={acc:.4f}\')\n    save_results(\'classifi

In [70]:
'''df = df[cols+['d']]
df'''

"df = df[cols+['d']]\ndf"

Hacemos una función que entrene el modelo, lo valide utilizando walk-forward y calcule el accuracy.

En esta versión de la función se usa una ventana de 90 días usando las fechas de los índices.

In [71]:
'''import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=50):
    # Definir el período de test basado en la frecuencia
    if freq == '1h': total_period = pd.Timedelta(days=7)
    elif freq == '4h': total_period = pd.Timedelta(days=15)
    else: total_period = pd.Timedelta(days=90)
    test_period = total_period * 0.3   # 30% de los días
    train_period = total_period - test_period  # 63 días de entrenamiento
    
    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }
        
        results = []
        for ric in data:
            df, cols = dfs[ric]
            df = df[cols + ['d']]
            df['timestamp'] = pd.to_datetime(df.index)
            
            max_time = df['timestamp'].max()
            min_time = df['timestamp'].min() + total_period
            split_dates = []
            current_time = min_time
            while current_time < max_time:
                split_dates.append(current_time)
                current_time += total_period  # Avance progresivo con total period
            
            for split_date in split_dates:
                train = df[(df['timestamp'] >= split_date - train_period) & (df['timestamp'] < split_date)]
                test = df[(df['timestamp'] >= split_date) & (df['timestamp'] < split_date + test_period)]
                
                if len(test) == 0:
                    continue  # Evitar iteraciones con test vacío
                
                X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']
                
                # Normalizar usando solo datos de entrenamiento
                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std
                
                # Crear y entrenar el modelo
                model = model_class(**trial_params)
                model.fit(X_train, y_train)
                
                # Predicción y evaluación
                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)
        
        avg_acc = np.mean(results) if results else 0  # Optuna maximiza esta métrica
        print(f'OUT-OF-SAMPLE | {ric:7s} | acc={avg_acc:.4f}')
        return avg_acc
    
    # Optimización con Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print("Mejores parámetros encontrados:", study.best_params)
    return study.best_params'''

'import optuna\nfrom sklearn.model_selection import TimeSeriesSplit\nfrom sklearn.metrics import accuracy_score\n\ndef walk_forward_fit_test(model_class, freq, model_params={}, n_trials=50):\n    # Definir el período de test basado en la frecuencia\n    if freq == \'1h\': total_period = pd.Timedelta(days=7)\n    elif freq == \'4h\': total_period = pd.Timedelta(days=15)\n    else: total_period = pd.Timedelta(days=90)\n    test_period = total_period * 0.3   # 30% de los días\n    train_period = total_period - test_period  # 63 días de entrenamiento\n    \n    def objective(trial):\n        trial_params = {\n            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),\n            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),\n            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),\n            "max_iter": model_params.get("max_iter", 1000),\n            "early_stopping": model_params.get("early_stopping", True),\n  

Modelo MLP Classifier

In [72]:
def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else: period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        results = []
        for ric in data:
            df, cols = dfs[ric]
            df = df[cols + ['d']]
            df['timestamp'] = pd.to_datetime(df.index)
            max_time = df['timestamp'].max()
            cutoff = max_time - final_test_period  # Reservamos el último año
            cutoff -= pd.Timedelta(days=window_pred)

            df_trainval = df[df['timestamp'] < cutoff]

            # Walk-forward en datos previos al último año
            min_time = df_trainval['timestamp'].min()
            split_dates = []
            current_time = min_time + period
            while current_time < cutoff:
                split_dates.append(current_time)
                current_time += period
            split_dates = split_dates[-5:]

            for split_date in split_dates:
                train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
                test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]
                fin_train = split_date - pd.Timedelta(days=window_pred)
                print('split_date, comienzo split', split_date)
                fin_test= split_date + period
                print('fin_train', fin_train - pd.Timedelta(days=window_pred))
                print('fin_test', fin_test )
                print('\n')

                if len(test) == 0:
                    continue

                X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std

                model = model_class(**trial_params)
                model.fit(X_train, y_train)

                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)

            avg_acc = np.mean(results)
            print(f'VALIDATION | {ric:7s} | acc={avg_acc:.4f}')
            save_results(model_class.__name__, ric, avg_acc, "OUT-SAMPLE")
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con mejor hiperparámetros, test en último año
    for ric in data:
        df, cols = dfs[ric]
        df = df[cols + ['d']]
        df['timestamp'] = pd.to_datetime(df.index)
        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period

        train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
        test = df[df['timestamp'] >= cutoff]
        

        if len(test) == 0:
            continue

        X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
        X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

        mean, std = X_train.mean(), X_train.std()
        std.replace(0, 1, inplace=True)
        X_train = (X_train - mean) / std
        X_test = (X_test - mean) / std

        model = model_class(
            hidden_layer_sizes=(best_params["hidden_units"],),
            alpha=best_params["alpha"],
            learning_rate_init=best_params["learning_rate"],
            max_iter=model_params.get("max_iter", 1000),
            early_stopping=model_params.get("early_stopping", True),
            validation_fraction=model_params.get("validation_fraction", 0.15),
            shuffle=model_params.get("shuffle", False),
            random_state=model_params.get("random_state", 100),
        )
        model.fit(X_train, y_train)
        pred = np.where(model.predict(X_test) > 0.5, 1, 0)
        acc = accuracy_score(y_test, pred)
        print(f'FINAL TEST | {ric:7s} | acc={acc:.4f}')
        save_results(model_class.__name__, ric, acc, "FINAL-TEST")

    return best_params


In [73]:
      
# Ejecutar la optimización
model_params = {
    "max_iter": 1000,
    "early_stopping": True,
    "validation_fraction": 0.15,
    "shuffle": False,
    "random_state": 100
}

tuned_params = walk_forward_fit_test(MLPClassifier, frequency, model_params, n_trials=1)


[I 2025-04-22 18:10:42,935] A new study created in memory with name: no-name-edebd4ec-3ff7-4ab5-ad49-1b3c2bcc24bb
c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:699: UserWarning: The distribution is specified by [32, 1024] and step=64, but the range is not divisible by `step`. It will be replaced by [32, 992].
  warnings.warn(
C:\Users\raque\AppData\Local\Temp\ipykernel_28176\2666905343.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
C:\Users\raque\AppData\Local\Temp\ipykernel_28176\2666905343.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate_

split_date, comienzo split 2022-10-17 00:00:00
fin_train 2022-10-03 00:00:00
fin_test 2023-01-15 00:00:00


split_date, comienzo split 2023-01-15 00:00:00
fin_train 2023-01-01 00:00:00
fin_test 2023-04-15 00:00:00


split_date, comienzo split 2023-04-15 00:00:00
fin_train 2023-04-01 00:00:00
fin_test 2023-07-14 00:00:00


split_date, comienzo split 2023-07-14 00:00:00
fin_train 2023-06-30 00:00:00
fin_test 2023-10-12 00:00:00


split_date, comienzo split 2023-10-12 00:00:00
fin_train 2023-09-28 00:00:00
fin_test 2024-01-10 00:00:00


VALIDATION | BTCUSDT | acc=0.7276
split_date, comienzo split 2022-10-17 00:00:00
fin_train 2022-10-03 00:00:00
fin_test 2023-01-15 00:00:00




C:\Users\raque\AppData\Local\Temp\ipykernel_28176\2666905343.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df.index)


split_date, comienzo split 2023-01-15 00:00:00
fin_train 2023-01-01 00:00:00
fin_test 2023-04-15 00:00:00


split_date, comienzo split 2023-04-15 00:00:00
fin_train 2023-04-01 00:00:00
fin_test 2023-07-14 00:00:00


split_date, comienzo split 2023-07-14 00:00:00
fin_train 2023-06-30 00:00:00
fin_test 2023-10-12 00:00:00


split_date, comienzo split 2023-10-12 00:00:00
fin_train 2023-09-28 00:00:00
fin_test 2024-01-10 00:00:00


VALIDATION | AVAXUSDT | acc=0.7779
split_date, comienzo split 2022-10-17 00:00:00
fin_train 2022-10-03 00:00:00
fin_test 2023-01-15 00:00:00




C:\Users\raque\AppData\Local\Temp\ipykernel_28176\2666905343.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df.index)


...

FINAL TEST | BTCUSDT | acc=0.8224


C:\Users\raque\AppData\Local\Temp\ipykernel_28176\2666905343.py:84: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df.index)


FINAL TEST | AVAXUSDT | acc=0.8224


C:\Users\raque\AppData\Local\Temp\ipykernel_28176\2666905343.py:84: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['timestamp'] = pd.to_datetime(df.index)


FINAL TEST | ETHUSDT | acc=0.8087


Modelo Bagging Classifier

In [74]:
'''import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=50):
    # Definir el período de test basado en la frecuencia
    if freq == '1h': total_period = pd.Timedelta(days=7)
    elif freq == '4h': total_period = pd.Timedelta(days=15)
    else: total_period = pd.Timedelta(days=90)
    test_period = total_period * 0.3   # 30% de los días
    train_period = total_period - test_period  # 63 días de entrenamiento
    
    def objective(trial):
        trial_params = {
            "n_estimators": trial.suggest_int("n_estimators", 10, 100, step=10),
            "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
            "max_features": trial.suggest_float("max_features", 0.5, 1.0),
            "random_state": model_params.get("random_state", 100)
        }
        
        results = []
        for ric in data:
            df, cols = dfs[ric]
            df = df[cols + ['d']]
            df['timestamp'] = pd.to_datetime(df.index)
            
            max_time = df['timestamp'].max()
            min_time = df['timestamp'].min() + total_period
            split_dates = []
            current_time = min_time
            while current_time < max_time:
                split_dates.append(current_time)
                current_time += total_period  # Avance progresivo con total period
            
            for split_date in split_dates:
                train = df[(df['timestamp'] >= split_date - train_period) & (df['timestamp'] < split_date)]
                test = df[(df['timestamp'] >= split_date) & (df['timestamp'] < split_date + test_period)]
                
                if len(test) == 0:
                    continue  # Evitar iteraciones con test vacío
                
                X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']
                
                # Normalizar usando solo datos de entrenamiento
                mean, std = X_train.mean(), X_train.std()
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std
                
                # Crear y entrenar el modelo
                model = model_class(**trial_params)
                model.fit(X_train, y_train)
                
                # Predicción y evaluación
                pred = model.predict(X_test)
                acc = accuracy_score(y_test, pred)
                results.append(acc)
        
        avg_acc = np.mean(results) if results else 0  # Optuna maximiza esta métrica
        print(f'OUT-OF-SAMPLE | {ric:7s} | acc={avg_acc:.4f}')
        return avg_acc
    
    # Optimización con Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print("Mejores parámetros encontrados:", study.best_params)
    return study.best_params
'''

'import optuna\nfrom sklearn.model_selection import TimeSeriesSplit\nfrom sklearn.metrics import accuracy_score\n\ndef walk_forward_fit_test(model_class, freq, model_params={}, n_trials=50):\n    # Definir el período de test basado en la frecuencia\n    if freq == \'1h\': total_period = pd.Timedelta(days=7)\n    elif freq == \'4h\': total_period = pd.Timedelta(days=15)\n    else: total_period = pd.Timedelta(days=90)\n    test_period = total_period * 0.3   # 30% de los días\n    train_period = total_period - test_period  # 63 días de entrenamiento\n    \n    def objective(trial):\n        trial_params = {\n            "n_estimators": trial.suggest_int("n_estimators", 10, 100, step=10),\n            "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),\n            "max_features": trial.suggest_float("max_features", 0.5, 1.0),\n            "random_state": model_params.get("random_state", 100)\n        }\n        \n        results = []\n        for ric in data:\n            df, col

In [75]:
'''from sklearn.ensemble import BaggingClassifier
from sklearn.neural_network import MLPClassifier   

# Definir base_estimator
base_estimator = MLPClassifier(tuned_params) 

# Ejecutar la optimización
tuned_params_b = walk_forward_fit_test(BaggingClassifier, frequency, {"base_estimator": base_estimator},  
                            n_trials=5)
'''

'from sklearn.ensemble import BaggingClassifier\nfrom sklearn.neural_network import MLPClassifier   \n\n# Definir base_estimator\nbase_estimator = MLPClassifier(tuned_params) \n\n# Ejecutar la optimización\ntuned_params_b = walk_forward_fit_test(BaggingClassifier, frequency, {"base_estimator": base_estimator},  \n                            n_trials=5)\n'

TEST

In [76]:
'''
def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else: period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        acc_por_ric = {}

        try:
            for ric in data:
                df, cols = dfs[ric]
                df = df[cols + ['d']]
                df['timestamp'] = pd.to_datetime(df.index)
                max_time = df['timestamp'].max()
                cutoff = max_time - final_test_period
                df_trainval = df[df['timestamp'] < cutoff]

                min_time = df_trainval['timestamp'].min()
                split_dates = []
                current_time = min_time + period
                while current_time < cutoff:
                    split_dates.append(current_time)
                    current_time += period
                split_dates = split_dates[-5:]

                results = []
                for split_date in split_dates:
                    train = df_trainval[df_trainval['timestamp'] < split_date]
                    test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

                    if len(test) == 0:
                        continue

                    X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                    X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

                    mean, std = X_train.mean(), X_train.std()
                    std.replace(0, 1, inplace=True)
                    X_train = (X_train - mean) / std
                    X_test = (X_test - mean) / std

                    model = model_class(**trial_params)
                    model.fit(X_train, y_train)

                    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                    acc = accuracy_score(y_test, pred)
                    results.append(acc)

                if results:
                    avg_acc = np.mean(results)
                    acc_por_ric[ric] = avg_acc

            # Ahora imprimimos solo una vez por modelo (no por trial)
            for ric, avg_acc in acc_por_ric.items():
                print(f'OUT-OF-SAMPLE | {ric:7s} | acc={avg_acc:.4f}')
                save_results(model_class.__name__, ric, avg_acc, "OUT-SAMPLE")

            # Retornamos el promedio global del modelo en todos los activos
            return np.mean(list(acc_por_ric.values())) if acc_por_ric else 0.0

        except Exception as e:
            print(f"Trial failed with exception: {e}")
            return 0.0


    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con mejor hiperparámetros, test en último año
    for ric in data:
        df, cols = dfs[ric]
        df = df[cols + ['d']]
        df['timestamp'] = pd.to_datetime(df.index)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period

        train = df[df['timestamp'] < cutoff]
        test = df[df['timestamp'] >= cutoff]

        if len(test) == 0:
            continue

        X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
        X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']

        mean, std = X_train.mean(), X_train.std()
        std.replace(0, 1, inplace=True)
        X_train = (X_train - mean) / std
        X_test = (X_test - mean) / std

        model = model_class(
            hidden_layer_sizes=(best_params["hidden_units"],),
            alpha=best_params["alpha"],
            learning_rate_init=best_params["learning_rate"],
            max_iter=model_params.get("max_iter", 1000),
            early_stopping=model_params.get("early_stopping", True),
            validation_fraction=model_params.get("validation_fraction", 0.15),
            shuffle=model_params.get("shuffle", False),
            random_state=model_params.get("random_state", 100),
        )
        model.fit(X_train, y_train)
        pred = np.where(model.predict(X_test) > 0.5, 1, 0)
        acc = accuracy_score(y_test, pred)
        print(f'FINAL TEST | {ric:7s} | acc={acc:.4f}')
        save_results(model_class.__name__, ric, acc, "FINAL-TEST")

    return best_params
'''

'\ndef walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):\n    if freq == \'1h\':\n        period = pd.Timedelta(days=7)\n    elif freq == \'4h\':\n        period = pd.Timedelta(days=15)\n    else: period = pd.Timedelta(days=90)\n    final_test_period = pd.Timedelta(days=365)\n\n    def objective(trial):\n        trial_params = {\n            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),\n            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),\n            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),\n            "max_iter": model_params.get("max_iter", 1000),\n            "early_stopping": model_params.get("early_stopping", True),\n            "validation_fraction": model_params.get("validation_fraction", 0.15),\n            "shuffle": model_params.get("shuffle", False),\n            "random_state": model_params.get("random_state", 100),\n        }\n\n        acc_por_ric = {}\n\n        tr

Modelo Global

In [77]:
'''def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)

    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        try:
            trial_params = {
                "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
                "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                "learning_rate_init": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),
                "max_iter": model_params.get("max_iter", 1000),
                "early_stopping": model_params.get("early_stopping", True),
                "validation_fraction": model_params.get("validation_fraction", 0.15),
                "shuffle": model_params.get("shuffle", False),
                "random_state": model_params.get("random_state", 100),
            }

            results = []
            for split_date in get_split_dates(period, final_test_period):
                global_train, global_test = [], []

                for ric in data:
                    df, cols = dfs[ric]
                    df = df[cols + ['d']]
                    df['timestamp'] = pd.to_datetime(df.index)
                    cutoff = df['timestamp'].max() - final_test_period
                    df_trainval = df[df['timestamp'] < cutoff]

                    train = df_trainval[df_trainval['timestamp'] < split_date]
                    test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

                    if len(test) == 0 or len(train) == 0:
                        continue

                    global_train.append(train)
                    global_test.append(test)

                if not global_train or not global_test:
                    print("[Trial Skipped] No se pudo generar train/test global.")
                    return None

                train_df = pd.concat(global_train)
                test_df = pd.concat(global_test)

                X_train, y_train = train_df.drop(columns=['d', 'timestamp']), train_df['d']
                X_test, y_test = test_df.drop(columns=['d', 'timestamp']), test_df['d']

                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std

                X_train = X_train.fillna(X_train.mean())
                X_test = X_test.fillna(X_train.mean())

                model = model_class(**trial_params)
                model.fit(X_train, y_train)

                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)

            if not results:
                print("[Trial Skipped] No se generaron métricas.")
                return None

            avg_acc = np.mean(results)
            print(f'[GLOBAL MODEL] acc={avg_acc:.4f}')
            save_results(model_class.__name__, "GLOBAL", acc, "HIPERPARAM-TRAIN")
            return avg_acc

        except Exception as e:
            print(f"[Trial Failed] {e}")
            return None

    def get_split_dates(period, final_test_period):
        all_timestamps = [pd.to_datetime(dfs[ric][0].index) for ric in data]
        min_time = max(min(ts) for ts in all_timestamps)
        max_time = min(max(ts) for ts in all_timestamps)
        cutoff = max_time - final_test_period

        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        return split_dates[-5:]

    # Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # ENTRENAMIENTO FINAL
    global_train, global_test = [], []
    for ric in data:
        df, cols = dfs[ric]
        df = df[cols + ['d']]
        df['timestamp'] = pd.to_datetime(df.index)

        cutoff = df['timestamp'].max() - final_test_period
        train = df[df['timestamp'] < cutoff]
        test = df[df['timestamp'] >= cutoff]

        if len(test) == 0:
            continue

        global_train.append(train)
        global_test.append(test)

    train_df = pd.concat(global_train)
    test_df = pd.concat(global_test)

    X_train, y_train = train_df.drop(columns=['d', 'timestamp']), train_df['d']
    X_test, y_test = test_df.drop(columns=['d', 'timestamp']), test_df['d']

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std
    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_train.mean())

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)
    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)

    print(f'[GLOBAL FINAL TEST] acc={acc:.4f}')
    save_results(model_class.__name__, "GLOBAL", acc, "FINAL-TEST")

    return best_params
'''

'def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):\n    if freq == \'1h\':\n        period = pd.Timedelta(days=7)\n    elif freq == \'4h\':\n        period = pd.Timedelta(days=15)\n    else:\n        period = pd.Timedelta(days=90)\n\n    final_test_period = pd.Timedelta(days=365)\n\n    def objective(trial):\n        try:\n            trial_params = {\n                "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),\n                "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),\n                "learning_rate_init": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),\n                "max_iter": model_params.get("max_iter", 1000),\n                "early_stopping": model_params.get("early_stopping", True),\n                "validation_fraction": model_params.get("validation_fraction", 0.15),\n                "shuffle": model_params.get("shuffle", False),\n                "random_state": model_params.get("rand

Hace el walk-forwark añadiendo ventanas (period) que seconvierten en la parte de test y todo lo anterior es train.(SIN TEST)

In [ ]:
'''
def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5): 
    # Definir el período de test basado en la frecuencia
    if freq == '1h': period = pd.Timedelta(days=7)
    elif freq == '4h': period = pd.Timedelta(days=15)
    else: period = pd.Timedelta(days=90)
    
    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }
        
        results = []
        for ric in data:
            df, cols = dfs[ric]
            df = df[cols + ['d']]
            df['timestamp'] = pd.to_datetime(df.index)
            
            max_time = df['timestamp'].max()
            min_time = df['timestamp'].min()
            
            split_dates = []
            current_time = min_time + period
            while current_time < max_time:
                split_dates.append(current_time)
                current_time += period
            split_dates = split_dates[-5:]  # Elegimos solo los últimos 5 splits
            
            for split_date in split_dates:
                train = df[df['timestamp'] < split_date]
                test = df[(df['timestamp'] >= split_date) & (df['timestamp'] < split_date + period)]
                
                if len(test) == 0:
                    continue  # Evitar iteraciones con test vacío
                
                X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']
                
                # Normalizar usando solo datos de entrenamiento
                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std
                
                # Crear y entrenar el modelo
                model = model_class(**trial_params)
                model.fit(X_train, y_train)
                
                # Predicción y evaluación
                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)
        avg_acc = np.mean(results)  
        print(f'OUT-OF-SAMPLE | {ric:7s} | acc={avg_acc:.4f}')
        save_results(model_class.__name__, ric, avg_acc, "OUT-SAMPLE")
        return avg_acc # Optuna maximiza esta métrica
        
    
    # Optimización con Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)
    
    print("Mejores parámetros encontrados:", study.best_params)
    return study.best_params'''

'# versión vieja, con mejores resultados?\n\n\ndef walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5): \n    # Definir el período de test basado en la frecuencia\n    if freq == \'1h\': period = pd.Timedelta(days=7)\n    elif freq == \'4h\': period = pd.Timedelta(days=15)\n    else: period = pd.Timedelta(days=90)\n    \n    def objective(trial):\n        trial_params = {\n            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),\n            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),\n            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),\n            "max_iter": model_params.get("max_iter", 1000),\n            "early_stopping": model_params.get("early_stopping", True),\n            "validation_fraction": model_params.get("validation_fraction", 0.15),\n            "shuffle": model_params.get("shuffle", False),\n            "random_state": model_params.get("random_state", 100),\n        }\

bagging

In [79]:
'''def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5): 
    # Definir el período de test basado en la frecuencia
    if freq == '1h': period = pd.Timedelta(days=7)
    elif freq == '4h': period = pd.Timedelta(days=15)
    else: period = pd.Timedelta(days=90)
    
    def objective(trial):
        trial_params = {
            "n_estimators": trial.suggest_int("n_estimators", 10, 100, step=10),
            "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
            "max_features": trial.suggest_float("max_features", 0.5, 1.0),
            "random_state": model_params.get("random_state", 100)
        }
        
        results = []
        for ric in data:
            df, cols = dfs[ric]
            df = df[cols + ['d']]
            df['timestamp'] = pd.to_datetime(df.index)
            
            max_time = df['timestamp'].max()
            min_time = df['timestamp'].min()
            
            split_dates = []
            current_time = min_time + period
            while current_time < max_time:
                split_dates.append(current_time)
                current_time += period
            
            for split_date in split_dates:
                train = df[df['timestamp'] < split_date]
                test = df[(df['timestamp'] >= split_date) & (df['timestamp'] < split_date + period)]
                
                if len(test) == 0:
                    continue  # Evitar iteraciones con test vacío
                
                X_train, y_train = train.drop(columns=['d', 'timestamp']), train['d']
                X_test, y_test = test.drop(columns=['d', 'timestamp']), test['d']
                
                # Normalizar usando solo datos de entrenamiento
                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std

                # Crear y entrenar el modelo
                model = model_class(**trial_params)
                model.fit(X_train, y_train)
                
                # Predicción y evaluación
                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)
        avg_acc = np.mean(results)  
        print(f'OUT-OF-SAMPLE | {ric:7s} | acc={avg_acc:.4f}')
        save_results(model_class.__name__, ric, avg_acc, "OUT-SAMPLE")
        return avg_acc # Optuna maximiza esta métrica
        
    
    # Optimización con Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)
    
    print("Mejores parámetros encontrados:", study.best_params)
    return study.best_params'''

'def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5): \n    # Definir el período de test basado en la frecuencia\n    if freq == \'1h\': period = pd.Timedelta(days=7)\n    elif freq == \'4h\': period = pd.Timedelta(days=15)\n    else: period = pd.Timedelta(days=90)\n    \n    def objective(trial):\n        trial_params = {\n            "n_estimators": trial.suggest_int("n_estimators", 10, 100, step=10),\n            "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),\n            "max_features": trial.suggest_float("max_features", 0.5, 1.0),\n            "random_state": model_params.get("random_state", 100)\n        }\n        \n        results = []\n        for ric in data:\n            df, cols = dfs[ric]\n            df = df[cols + [\'d\']]\n            df[\'timestamp\'] = pd.to_datetime(df.index)\n            \n            max_time = df[\'timestamp\'].max()\n            min_time = df[\'timestamp\'].min()\n            \n            split_dates = []\n  